# LangGraph course-generation agent: executable walkthrough

This notebook is a **learning and testing copy** of `ai/workflow.py`. The production
Python files remain unchanged and continue to be the source used by Django and Celery.

The agent researches a course topic, designs and validates a curriculum blueprint,
pauses for human approval, generates and validates the complete course package,
pauses for final approval, and persists the approved course.

> Important: the copied nodes perform real database writes. Execution is opt-in in the
> testing section near the end; read the safety notes before enabling it.


## 1. Notebook and Django setup

The notebook lives inside `apps/api/ai`, but Jupyter may start with a different working
directory. This cell finds the API project, adds it to Python's import path, selects
the Django settings module, and initialises Django before importing any models.

Run this cell first. It does not start a generation job or write application data.


In [ ]:
import os
import sys
from pathlib import Path

candidate_roots = [
    Path.cwd(),
    Path.cwd() / "apps" / "api",
    Path.cwd().parent,
    Path.cwd().parent.parent,
]
API_ROOT = next(
    (path.resolve() for path in candidate_roots if (path / "manage.py").exists()),
    None,
)
if API_ROOT is None:
    raise RuntimeError("Could not find apps/api (the directory containing manage.py).")

os.chdir(API_ROOT)
if str(API_ROOT) not in sys.path:
    sys.path.insert(0, str(API_ROOT))
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "config.settings")

import django

django.setup()
print(f"Django initialised from: {API_ROOT}")


## 2. Imports used by the copied workflow

These are the production workflow's imports, adjusted only from package-relative
imports such as `.models` to notebook-safe absolute imports such as `ai.models`.


In [ ]:
from contextlib import contextmanager, nullcontext
from urllib.parse import quote
from django.conf import settings
from django.db import transaction
from django.utils import timezone
from django.utils.module_loading import import_string
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.checkpoint.postgres import PostgresSaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt
from pydantic import ValidationError
from typing_extensions import TypedDict
from ai.blueprints import CurriculumBlueprint
from ai.models import (
    GeneratedArtifact,
    GenerationJob,
    ResearchFinding,
    ResearchQuestion,
    ResearchSource,
)
from ai.packages import GeneratedCoursePackage
from ai.persistence import persist_approved_course
from ai.research import PlannedResearchQuestion, ResearchProviderResult


## 3. Shared graph state

`BlueprintState` is the contract passed between nodes. It is `total=False`, so fields
can be added gradually. Each node returns a partial dictionary; LangGraph merges those
values into the current state.

The in-memory checkpointer is useful for local tests. Production can instead use the
PostgreSQL checkpointer selected by `workflow_checkpointer()`.


In [ ]:
_memory_checkpointer = InMemorySaver()

class BlueprintState(TypedDict, total=False):
    job_id: str
    blueprint: dict
    artifact_id: str
    validation_errors: list[dict]
    revision_count: int
    feedback: str
    course_package: dict
    package_artifact_id: str
    package_validation_errors: list[dict]
    package_revision_count: int
    package_feedback: str
    research_questions: list[dict]
    research_findings: list[dict]


## 4. Checkpoint and context helpers

These helpers are not graph nodes, but they supply essential infrastructure:

- `_database_url` selects the configured LangGraph/PostgreSQL connection.
- `workflow_checkpointer` uses PostgreSQL when available and memory otherwise.
- `_source_context` serialises ready document chunks for generators/providers.
- `_research_context` exposes only findings meeting quality thresholds.


In [ ]:
def _database_url():
    if settings.LANGGRAPH_DATABASE_URL:
        return settings.LANGGRAPH_DATABASE_URL
    database = settings.DATABASES['default']
    if 'postgresql' not in database['ENGINE']:
        return ''
    user = quote(str(database.get('USER') or ''), safe='')
    password = quote(str(database.get('PASSWORD') or ''), safe='')
    host = database.get('HOST') or 'localhost'
    port = database.get('PORT') or '5432'
    name = quote(str(database.get('NAME') or ''), safe='')
    return f'postgresql://{user}:{password}@{host}:{port}/{name}'


In [ ]:
def workflow_checkpointer():
    database_url = _database_url()
    if not database_url:
        with nullcontext(_memory_checkpointer) as checkpointer:
            yield checkpointer
        return
    with PostgresSaver.from_conn_string(database_url) as checkpointer:
        checkpointer.setup()
        yield checkpointer


In [ ]:
def _source_context(job):
    return [
        {
            'document_id': str(chunk.document_id),
            'chunk_id': str(chunk.id),
            'filename': chunk.document.original_filename,
            'page_number': chunk.page_number,
            'heading': chunk.heading,
            'content': chunk.content,
        }
        for chunk in job.source_documents.filter(status='READY')
        .prefetch_related('chunks')
        .all()
        for chunk in chunk.chunks.all()
    ]


In [ ]:
def _research_context(job):
    return [
        {
            'question': finding.question.query,
            'claim': finding.claim,
            'evidence': finding.evidence,
            'confidence': finding.confidence,
            'source': {
                'id': str(finding.source_id),
                'type': finding.source.type,
                'title': finding.source.title,
                'url': finding.source.url,
                'publisher': finding.source.publisher,
                'published_at': (
                    finding.source.published_at.isoformat()
                    if finding.source.published_at else None
                ),
                'reliability_score': finding.source.reliability_score,
            },
            'source_locator': finding.source_locator,
        }
        for finding in ResearchFinding.objects.filter(
            question__job=job,
            confidence__gte=settings.RESEARCH_MIN_CONFIDENCE,
            source__reliability_score__gte=settings.RESEARCH_MIN_RELIABILITY,
        )
        .select_related('question', 'source')
        .all()
    ]


## 5. Workflow nodes and routing functions

The following sections present every node in execution order. Each Markdown cell
explains its responsibility, inputs, outputs, and side effects before the copied code.


## Node: `plan_research`

This is the workflow's first real node. It loads the generation job and its uploaded
document chunks, marks the job as researching, calls the configured research planner,
validates every proposed question, and stores those questions in the database.

**Reads:** `job_id`  
**Writes:** `research_questions`


In [ ]:
def plan_research(state: BlueprintState):
    job = GenerationJob.objects.prefetch_related('source_documents__chunks').get(
        pk=state['job_id']
    )
    job.status = GenerationJob.Status.RESEARCHING
    job.current_stage = 'planning_research'
    job.progress_percent = 18
    job.status_message = 'Planning research for the curriculum.'
    job.heartbeat_at = timezone.now()
    job.save()
    planner = import_string(settings.COURSE_RESEARCH_PLANNER)
    raw_questions = planner(brief=job.course_brief, documents=_source_context(job))
    questions = [
        PlannedResearchQuestion.model_validate(question).model_dump(mode='json')
        for question in raw_questions
    ]
    with transaction.atomic():
        job.research_questions.all().delete()
        ResearchQuestion.objects.bulk_create(
            [
                ResearchQuestion(
                    job=job,
                    query=question['query'],
                    rationale=question['rationale'],
                    priority=question['priority'],
                )
                for question in questions
            ]
        )
        job.add_event(
            'RESEARCH_PLANNED',
            f'Planned {len(questions)} curriculum research questions.',
        )
    return {'research_questions': questions}


## Node: `execute_research`

This node sends the planned questions, course brief, and document chunks to the
configured research provider. It validates the provider response, persists sources
and findings, marks each question as completed or having no results, and returns only
research that passes the configured reliability and confidence thresholds.

**Reads:** `job_id`, `research_questions`  
**Writes:** `research_findings`


In [ ]:
def execute_research(state: BlueprintState):
    job = GenerationJob.objects.prefetch_related('source_documents__chunks').get(
        pk=state['job_id']
    )
    job.current_stage = 'executing_research'
    job.progress_percent = 24
    job.status_message = 'Collecting and validating curriculum research.'
    job.heartbeat_at = timezone.now()
    job.save()
    provider = import_string(settings.COURSE_RESEARCH_PROVIDER)
    raw_result = provider(
        job_id=str(job.id),
        questions=state['research_questions'],
        brief=job.course_brief,
        documents=_source_context(job),
    )
    result = ResearchProviderResult.model_validate(raw_result)
    with transaction.atomic():
        job.research_sources.all().delete()
        source_by_key = {}
        for source_data in result.sources:
            url = str(source_data.url) if source_data.url else ''
            source = ResearchSource.objects.create(
                job=job,
                type=source_data.type,
                canonical_uri=url or source_data.provider_key,
                url=url,
                title=source_data.title,
                publisher=source_data.publisher,
                authors=source_data.authors,
                published_at=source_data.published_at,
                reliability_score=source_data.reliability_score,
                metadata=source_data.metadata,
            )
            source_by_key[source_data.provider_key] = source
        question_by_query = {
            question.query: question for question in job.research_questions.all()
        }
        finding_counts = {query: 0 for query in question_by_query}
        persisted_findings = []
        for finding_data in result.findings:
            question = question_by_query.get(finding_data.question_query)
            if question is None:
                raise ValueError(
                    f'Research finding references an unknown question: {finding_data.question_query}'
                )
            finding = ResearchFinding.objects.create(
                question=question,
                source=source_by_key[finding_data.source_key],
                claim=finding_data.claim,
                evidence=finding_data.evidence,
                confidence=finding_data.confidence,
                source_locator=finding_data.source_locator,
            )
            finding_counts[question.query] += 1
            persisted_findings.append(str(finding.id))
        for query, question in question_by_query.items():
            question.status = (
                ResearchQuestion.Status.COMPLETED
                if finding_counts[query]
                else ResearchQuestion.Status.NO_RESULTS
            )
            question.save(update_fields=('status', 'updated_at'))
        job.add_event(
            'RESEARCH_COMPLETED',
            f'Validated {len(result.sources)} sources and {len(result.findings)} findings.',
            {
                'minimum_reliability': settings.RESEARCH_MIN_RELIABILITY,
                'minimum_confidence': settings.RESEARCH_MIN_CONFIDENCE,
            },
        )
    return {
        'research_findings': _research_context(job),
    }


## Node: `design_blueprint`

The curriculum generator receives the brief, source chunks, validated research,
review feedback, and any previous blueprint. Keeping the previous draft and feedback
in state is what lets this same node handle both initial creation and revisions.

**Reads:** `job_id`, optional `feedback`, optional `blueprint`  
**Writes:** `blueprint`, clears `validation_errors`


In [ ]:
def design_blueprint(state: BlueprintState):
    job = GenerationJob.objects.prefetch_related('source_documents__chunks').get(
        pk=state['job_id']
    )
    job.status = GenerationJob.Status.DESIGNING_CURRICULUM
    job.current_stage = 'designing_curriculum'
    job.progress_percent = 30
    job.status_message = 'Designing the curriculum blueprint from validated research.'
    job.heartbeat_at = timezone.now()
    job.save()
    generator = import_string(settings.CURRICULUM_BLUEPRINT_GENERATOR)
    blueprint = generator(
        job_id=str(job.id),
        brief=job.course_brief,
        sources=_source_context(job),
        research=_research_context(job),
        feedback=state.get('feedback', ''),
        previous_blueprint=state.get('blueprint'),
    )
    return {'blueprint': blueprint, 'validation_errors': []}


## Node: `validate_blueprint`

This is a deterministic quality gate. Pydantic validates the generated dictionary
against `CurriculumBlueprint`. A valid model is normalised back to JSON-compatible
data; validation problems are returned in state instead of immediately crashing.

**Reads:** `blueprint`  
**Writes:** normalised `blueprint`, `validation_errors`


In [ ]:
def validate_blueprint(state: BlueprintState):
    try:
        blueprint = CurriculumBlueprint.model_validate(state['blueprint'])
    except ValidationError as exc:
        return {'validation_errors': exc.errors(include_url=False)}
    return {'blueprint': blueprint.model_dump(mode='json'), 'validation_errors': []}


### Router: `validation_route`

This conditional edge chooses the next node after blueprint validation:

- valid → persist the blueprint;
- invalid with revision attempts remaining → automatically prepare a revision;
- still invalid after two revisions → fail explicitly.


In [ ]:
def validation_route(state: BlueprintState):
    if not state.get('validation_errors'):
        return 'persist_blueprint'
    if state.get('revision_count', 0) >= 2:
        return 'validation_failed'
    return 'automatic_revision'


## Node: `automatic_revision`

This node converts Pydantic validation errors into plain feedback for the generator
and increments the revision counter. The graph then loops back to `design_blueprint`.

**Reads:** `validation_errors`, `revision_count`  
**Writes:** `feedback`, `revision_count`


In [ ]:
def automatic_revision(state: BlueprintState):
    messages = [error.get('msg', 'Invalid blueprint') for error in state['validation_errors']]
    return {
        'feedback': 'Correct these validation errors: ' + '; '.join(messages),
        'revision_count': state.get('revision_count', 0) + 1,
    }


## Node: `validation_failed`

This terminal error node prevents an endlessly looping generator. Reaching it means
the blueprint remained structurally invalid after the allowed automatic corrections.


In [ ]:
def validation_failed(state: BlueprintState):
    raise ValueError('Blueprint remained invalid after two automatic revision attempts.')


## Node: `persist_blueprint`

The validated blueprint becomes a versioned `GeneratedArtifact`. Older draft
blueprints are superseded atomically, source-document lineage is recorded, and a
domain event announces that the new version is ready for human review.

**Reads:** `job_id`, `blueprint`, `revision_count`  
**Writes:** `artifact_id`


In [ ]:
def persist_blueprint(state: BlueprintState):
    job = GenerationJob.objects.get(pk=state['job_id'])
    version = state.get('revision_count', 0) + 1
    with transaction.atomic():
        GeneratedArtifact.objects.filter(
            job=job,
            type=GeneratedArtifact.Type.BLUEPRINT,
            status=GeneratedArtifact.Status.DRAFT,
        ).update(status=GeneratedArtifact.Status.SUPERSEDED)
        artifact, _ = GeneratedArtifact.objects.update_or_create(
            job=job,
            type=GeneratedArtifact.Type.BLUEPRINT,
            version=version,
            defaults={
                'status': GeneratedArtifact.Status.DRAFT,
                'content': state['blueprint'],
                'validation_errors': [],
                'source_document_ids': [
                    str(document_id)
                    for document_id in job.source_documents.filter(status='READY').values_list(
                        'id', flat=True
                    )
                ],
            },
        )
        job.add_event(
            'BLUEPRINT_READY',
            f'Curriculum blueprint version {version} is ready for review.',
            {'artifact_id': str(artifact.id), 'version': version},
        )
    return {'artifact_id': str(artifact.id)}


## Node: `review_blueprint` — human-in-the-loop pause

`interrupt(...)` checkpoints the graph and returns control to the application. When
the workflow is resumed, an `APPROVE` decision advances it; a `REVISE` decision stores
feedback and jumps back to blueprint design. Any other decision is rejected.

**Reads:** `artifact_id`, `blueprint`  
**Command result:** jump to `blueprint_approved` or `design_blueprint`


In [ ]:
def review_blueprint(state: BlueprintState):
    response = interrupt(
        {
            'kind': 'curriculum_blueprint_review',
            'artifact_id': state['artifact_id'],
            'blueprint': state['blueprint'],
        }
    )
    decision = response.get('decision') if isinstance(response, dict) else None
    artifact = GeneratedArtifact.objects.select_related('job').get(pk=state['artifact_id'])
    if decision == 'APPROVE':
        artifact.status = GeneratedArtifact.Status.APPROVED
        artifact.save(update_fields=('status', 'updated_at'))
        artifact.job.add_event(
            'BLUEPRINT_APPROVED',
            f'Curriculum blueprint version {artifact.version} was approved.',
            {'artifact_id': str(artifact.id)},
        )
        return Command(goto='blueprint_approved')
    if decision == 'REVISE':
        artifact.status = GeneratedArtifact.Status.REVISION_REQUESTED
        artifact.save(update_fields=('status', 'updated_at'))
        artifact.job.add_event(
            'BLUEPRINT_REVISION_REQUESTED',
            f'Revision requested for blueprint version {artifact.version}.',
            {'artifact_id': str(artifact.id)},
        )
        return Command(
            update={
                'feedback': str(response.get('feedback') or ''),
                'revision_count': state.get('revision_count', 0) + 1,
            },
            goto='design_blueprint',
        )
    raise ValueError('Blueprint review decision must be APPROVE or REVISE.')


## Node: `blueprint_approved`

This small transition node updates the job's operational status so users can see that
lesson generation has begun after blueprint approval.

**Reads:** `job_id`  
**Writes:** no graph-state fields; updates the database job


In [ ]:
def blueprint_approved(state: BlueprintState):
    job = GenerationJob.objects.get(pk=state['job_id'])
    job.status = GenerationJob.Status.GENERATING_LESSONS
    job.current_stage = 'generating_course_package'
    job.progress_percent = 45
    job.status_message = 'Curriculum blueprint approved; generating course content.'
    job.heartbeat_at = timezone.now()
    job.save()
    return {}


## Node: `generate_course_package`

The package generator expands the approved blueprint into lessons, assessments, and
flashcards. Like blueprint design, it receives previous output and review feedback so
the same node can generate a first draft or a revised package.

**Reads:** `job_id`, `blueprint`, optional package feedback/draft  
**Writes:** `course_package`, clears `package_validation_errors`


In [ ]:
def generate_course_package(state: BlueprintState):
    job = GenerationJob.objects.prefetch_related('source_documents__chunks').get(
        pk=state['job_id']
    )
    generator = import_string(settings.COURSE_PACKAGE_GENERATOR)
    package = generator(
        job_id=str(job.id),
        blueprint=state['blueprint'],
        sources=_source_context(job),
        research=_research_context(job),
        feedback=state.get('package_feedback', ''),
        previous_package=state.get('course_package'),
    )
    return {'course_package': package, 'package_validation_errors': []}


## Node: `validate_course_package`

This gate performs three checks: Pydantic schema validation, consistency with the
approved blueprint, and verification that every cited chunk belongs to a ready source
document for this job. This prevents invented or cross-job source references.

**Reads:** `job_id`, `course_package`, `blueprint`  
**Writes:** normalised `course_package`, `package_validation_errors`


In [ ]:
def validate_course_package(state: BlueprintState):
    try:
        package = GeneratedCoursePackage.model_validate(state['course_package'])
        package.validate_against_blueprint(state['blueprint'])
        valid_source_ids = {
            str(chunk_id)
            for chunk_id in GenerationJob.objects.get(pk=state['job_id'])
            .source_documents.filter(status='READY')
            .values_list('chunks__id', flat=True)
            if chunk_id is not None
        }
        referenced_source_ids = {
            chunk_id
            for module in package.modules
            for lesson in module.lessons
            for chunk_id in lesson.source_chunk_ids
        }
        unknown_source_ids = referenced_source_ids - valid_source_ids
        if unknown_source_ids:
            raise ValueError(
                f'Package references unknown source chunks: {sorted(unknown_source_ids)}'
            )
    except (ValidationError, ValueError) as exc:
        errors = exc.errors(include_url=False) if isinstance(exc, ValidationError) else [{'msg': str(exc)}]
        return {'package_validation_errors': errors}
    return {
        'course_package': package.model_dump(mode='json'),
        'package_validation_errors': [],
    }


### Router: `package_validation_route`

This is the package equivalent of the blueprint router: persist valid output, request
an automatic correction when attempts remain, or fail after the retry limit.


In [ ]:
def package_validation_route(state: BlueprintState):
    if not state.get('package_validation_errors'):
        return 'persist_course_package_artifact'
    if state.get('package_revision_count', 0) >= 2:
        return 'package_validation_failed'
    return 'automatic_package_revision'


## Node: `automatic_package_revision`

Validation messages are converted into generator feedback and the package revision
counter is incremented before looping back to package generation.

**Reads:** `package_validation_errors`, `package_revision_count`  
**Writes:** `package_feedback`, `package_revision_count`


In [ ]:
def automatic_package_revision(state: BlueprintState):
    messages = [
        error.get('msg', 'Invalid package') for error in state['package_validation_errors']
    ]
    return {
        'package_feedback': 'Correct these validation errors: ' + '; '.join(messages),
        'package_revision_count': state.get('package_revision_count', 0) + 1,
    }


## Node: `package_validation_failed`

This explicit failure stops repeated invalid package generation after two automatic
revision attempts.


In [ ]:
def package_validation_failed(state: BlueprintState):
    raise ValueError('Course package remained invalid after two automatic revision attempts.')


## Node: `persist_course_package_artifact`

The valid course package is stored as a versioned draft artifact. Previous drafts are
superseded, source lineage is attached, and an event signals that final review can
begin.

**Reads:** `job_id`, `course_package`, `package_revision_count`  
**Writes:** `package_artifact_id`


In [ ]:
def persist_course_package_artifact(state: BlueprintState):
    job = GenerationJob.objects.get(pk=state['job_id'])
    version = state.get('package_revision_count', 0) + 1
    with transaction.atomic():
        GeneratedArtifact.objects.filter(
            job=job,
            type=GeneratedArtifact.Type.COURSE_PACKAGE,
            status=GeneratedArtifact.Status.DRAFT,
        ).update(status=GeneratedArtifact.Status.SUPERSEDED)
        artifact, _ = GeneratedArtifact.objects.update_or_create(
            job=job,
            type=GeneratedArtifact.Type.COURSE_PACKAGE,
            version=version,
            defaults={
                'status': GeneratedArtifact.Status.DRAFT,
                'content': state['course_package'],
                'validation_errors': [],
                'source_document_ids': [
                    str(document_id)
                    for document_id in job.source_documents.filter(status='READY').values_list(
                        'id', flat=True
                    )
                ],
            },
        )
        job.add_event(
            'COURSE_PACKAGE_READY',
            f'Generated course package version {version} is ready for final review.',
            {'artifact_id': str(artifact.id), 'version': version},
        )
    return {'package_artifact_id': str(artifact.id)}


## Node: `review_course_package` — final human review

The second interrupt asks a reviewer to approve or revise the complete package.
Approval jumps to final persistence; revision records feedback and loops back to
package generation.

**Reads:** `package_artifact_id`, `course_package`  
**Command result:** jump to `persist_approved_course` or `generate_course_package`


In [ ]:
def review_course_package(state: BlueprintState):
    response = interrupt(
        {
            'kind': 'course_package_review',
            'artifact_id': state['package_artifact_id'],
            'course_package': state['course_package'],
        }
    )
    decision = response.get('decision') if isinstance(response, dict) else None
    artifact = GeneratedArtifact.objects.select_related('job').get(
        pk=state['package_artifact_id']
    )
    if decision == 'APPROVE':
        artifact.status = GeneratedArtifact.Status.APPROVED
        artifact.save(update_fields=('status', 'updated_at'))
        artifact.job.add_event(
            'COURSE_PACKAGE_APPROVED',
            f'Generated course package version {artifact.version} was approved.',
            {'artifact_id': str(artifact.id)},
        )
        return Command(goto='persist_approved_course')
    if decision == 'REVISE':
        artifact.status = GeneratedArtifact.Status.REVISION_REQUESTED
        artifact.save(update_fields=('status', 'updated_at'))
        artifact.job.add_event(
            'COURSE_PACKAGE_REVISION_REQUESTED',
            f'Revision requested for course package version {artifact.version}.',
            {'artifact_id': str(artifact.id)},
        )
        return Command(
            update={
                'package_feedback': str(response.get('feedback') or ''),
                'package_revision_count': state.get('package_revision_count', 0) + 1,
            },
            goto='generate_course_package',
        )
    raise ValueError('Course package review decision must be APPROVE or REVISE.')


## Node: `persist_approved_course_node`

This thin graph adapter calls the application's existing persistence service. That
service converts the approved package artifact into the LMS's permanent course,
module, lesson, quiz, and flashcard records.

**Reads:** `job_id`, `package_artifact_id`


In [ ]:
def persist_approved_course_node(state: BlueprintState):
    persist_approved_course(state['job_id'], state['package_artifact_id'])
    return {}


## 6. Assemble the graph

`build_blueprint_graph` registers each function under a stable node name and connects
the edges. Conditional edges implement validation branches, while `Command(goto=...)`
inside review nodes handles decisions that are known only after a human responds.

The high-level path is:

`START → research → blueprint → validate/revise → blueprint review → package → validate/revise → final review → persist → END`


In [ ]:
def build_blueprint_graph(checkpointer):
    builder = StateGraph(BlueprintState)
    builder.add_node('design_blueprint', design_blueprint)
    builder.add_node('plan_research', plan_research)
    builder.add_node('execute_research', execute_research)
    builder.add_node('validate_blueprint', validate_blueprint)
    builder.add_node('automatic_revision', automatic_revision)
    builder.add_node('validation_failed', validation_failed)
    builder.add_node('persist_blueprint', persist_blueprint)
    builder.add_node('review_blueprint', review_blueprint)
    builder.add_node('blueprint_approved', blueprint_approved)
    builder.add_node('generate_course_package', generate_course_package)
    builder.add_node('validate_course_package', validate_course_package)
    builder.add_node('automatic_package_revision', automatic_package_revision)
    builder.add_node('package_validation_failed', package_validation_failed)
    builder.add_node('persist_course_package_artifact', persist_course_package_artifact)
    builder.add_node('review_course_package', review_course_package)
    builder.add_node('persist_approved_course', persist_approved_course_node)
    builder.add_edge(START, 'plan_research')
    builder.add_edge('plan_research', 'execute_research')
    builder.add_edge('execute_research', 'design_blueprint')
    builder.add_edge('design_blueprint', 'validate_blueprint')
    builder.add_conditional_edges('validate_blueprint', validation_route)
    builder.add_edge('automatic_revision', 'design_blueprint')
    builder.add_edge('persist_blueprint', 'review_blueprint')
    builder.add_edge('blueprint_approved', 'generate_course_package')
    builder.add_edge('generate_course_package', 'validate_course_package')
    builder.add_conditional_edges('validate_course_package', package_validation_route)
    builder.add_edge('automatic_package_revision', 'generate_course_package')
    builder.add_edge('persist_course_package_artifact', 'review_course_package')
    builder.add_edge('persist_approved_course', END)
    return builder.compile(checkpointer=checkpointer)


## 7. Runtime helpers

These copied functions provide the production entry points:

- `_config` makes the job ID the durable LangGraph thread ID.
- `_record_graph_result` saves checkpoint metadata and maps interrupts to job status.
- `run_generation_workflow` starts a new graph execution.
- `resume_generation_workflow` resumes a paused graph with a review decision.


In [ ]:
def _config(job_id):
    return {'configurable': {'thread_id': str(job_id)}}


In [ ]:
def _record_graph_result(job_id, graph, config, result):
    snapshot = graph.get_state(config)
    checkpoint_id = snapshot.config.get('configurable', {}).get('checkpoint_id', '')
    job = GenerationJob.objects.get(pk=job_id)
    job.graph_thread_id = str(job_id)
    job.graph_checkpoint_id = checkpoint_id
    job.heartbeat_at = timezone.now()
    interrupts = result.get('__interrupt__') or []
    if interrupts:
        interrupt_value = getattr(interrupts[0], 'value', {})
        kind = interrupt_value.get('kind') if isinstance(interrupt_value, dict) else ''
        if kind == 'course_package_review':
            job.status = GenerationJob.Status.WAITING_FOR_FINAL_APPROVAL
            job.current_stage = 'course_package_review'
            job.progress_percent = 95
            job.status_message = 'Generated course package is waiting for final approval.'
        else:
            job.status = GenerationJob.Status.WAITING_FOR_BLUEPRINT_APPROVAL
            job.current_stage = 'blueprint_review'
            job.progress_percent = 40
            job.status_message = 'Curriculum blueprint is waiting for approval.'
        job.waiting_since = timezone.now()
        job.add_event(job.status, job.status_message)
    job.save()
    return {
        'job_id': str(job.id),
        'status': job.status,
        'checkpoint_id': checkpoint_id,
        'interrupted': bool(interrupts),
    }


In [ ]:
def run_generation_workflow(job_id):
    job = GenerationJob.objects.get(pk=job_id)
    job.status = GenerationJob.Status.RESEARCHING
    job.current_stage = 'starting_research'
    job.progress_percent = 15
    job.status_message = 'Preparing curriculum research.'
    job.heartbeat_at = timezone.now()
    job.save()
    config = _config(job_id)
    with workflow_checkpointer() as checkpointer:
        graph = build_blueprint_graph(checkpointer)
        result = graph.invoke(
            {
                'job_id': str(job_id),
                'revision_count': 0,
                'feedback': '',
                'package_revision_count': 0,
                'package_feedback': '',
            },
            config=config,
        )
        return _record_graph_result(job_id, graph, config, result)


In [ ]:
def resume_generation_workflow(job_id, review):
    config = _config(job_id)
    with workflow_checkpointer() as checkpointer:
        graph = build_blueprint_graph(checkpointer)
        result = graph.invoke(Command(resume=review), config=config)
        return _record_graph_result(job_id, graph, config, result)


## 8. Inspect the graph without running a job

This is a read-only structural test. It compiles the copied graph with an in-memory
checkpointer and prints a Mermaid diagram. In JupyterLab, the plain Mermaid text can
be pasted into a Mermaid viewer if it is not rendered automatically.


In [ ]:
inspection_graph = build_blueprint_graph(InMemorySaver())
print(inspection_graph.get_graph().draw_mermaid())


## 9. Choose an existing generation job

Use a real `GenerationJob` UUID from the current database. The lookup cell is
read-only and shows recent candidates. Starting the workflow is intentionally guarded
by `RUN_WORKFLOW = False` in the next section.

The job should have a valid course brief and all attached source documents must be in
the `READY` state.


In [ ]:
recent_jobs = list(
    GenerationJob.objects.order_by("-created_at").values(
        "id", "status", "current_stage", "progress_percent"
    )[:10]
)
recent_jobs


In [ ]:
JOB_ID = ""  # Paste a GenerationJob UUID here.

if JOB_ID:
    selected_job = GenerationJob.objects.get(pk=JOB_ID)
    print(
        selected_job.id,
        selected_job.status,
        selected_job.current_stage,
        selected_job.course_brief,
    )
else:
    print("Set JOB_ID before running a workflow test.")


## 10. Start the copied workflow (opt in)

Enabling this cell performs the real side effects described above: it updates the job,
creates research rows and artifacts, calls the configured providers/generators, and
stops at the blueprint-review interrupt.

Use a disposable test job. Keep `RUN_WORKFLOW` false until `JOB_ID` has been checked.


In [ ]:
RUN_WORKFLOW = False

if RUN_WORKFLOW:
    if not JOB_ID:
        raise ValueError("Set JOB_ID before enabling RUN_WORKFLOW.")
    first_result = run_generation_workflow(JOB_ID)
    print(first_result)
else:
    print("Workflow not started. Set RUN_WORKFLOW = True to opt in.")


## 11. Resume the blueprint review (opt in)

Run this only after the previous result reports an interrupt at `blueprint_review`.

- `APPROVE` continues to course-package generation and pauses at final review.
- `REVISE` loops back to blueprint design using the supplied feedback.

Checkpoint continuity requires the same configured PostgreSQL checkpointer used when
the workflow started. The in-memory fallback only survives within this Python process.


In [ ]:
RESUME_BLUEPRINT = False
blueprint_review = {
    "decision": "APPROVE",  # Or "REVISE".
    "feedback": "",
}

if RESUME_BLUEPRINT:
    if not JOB_ID:
        raise ValueError("Set JOB_ID before resuming.")
    blueprint_result = resume_generation_workflow(JOB_ID, blueprint_review)
    print(blueprint_result)
else:
    print("Blueprint review not submitted.")


## 12. Resume the final package review (opt in)

Run this only after the graph pauses at `course_package_review`.

- `APPROVE` persists the generated package as LMS course data.
- `REVISE` returns to package generation with reviewer feedback.

Approval is a material database action, so verify the artifact in the application
before enabling this cell.


In [ ]:
RESUME_PACKAGE = False
package_review = {
    "decision": "APPROVE",  # Or "REVISE".
    "feedback": "",
}

if RESUME_PACKAGE:
    if not JOB_ID:
        raise ValueError("Set JOB_ID before resuming.")
    package_result = resume_generation_workflow(JOB_ID, package_review)
    print(package_result)
else:
    print("Final package review not submitted.")


## 13. Suggested debugging approach

For focused testing, call deterministic nodes such as the validators with a small
state dictionary before attempting the whole graph. Database-backed nodes need a real
job and should be tested against disposable data.

Useful things to inspect after a pause are `graph.get_state(_config(JOB_ID)).values`,
the latest `GeneratedArtifact`, the job's events, and the persisted research sources
and findings. Automated production coverage remains in `ai/tests.py`; this notebook is
intended for exploration and onboarding rather than replacing those tests.
